In [7]:
import torch 
import torch.nn.functional as F
from transformers import AutoModel,AutoTokenizer,AutoModelForSeq2SeqLM
embedding_model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
generator_model_name="google/flan-t5-small"
embed_tokenizer=AutoTokenizer.from_pretrained(embedding_model_name)
embed_model=AutoModel.from_pretrained(embedding_model_name)
gen_tokenizer=AutoTokenizer.from_pretrained(generator_model_name)
gen_model=AutoModelForSeq2SeqLM.from_pretrained(generator_model_name)

documents=[
    "For fat loss dinner,choose high protien,low fat,and moderate carbs",
    "Good chest exercises include bench press,incline dumbbell press,and cable fly",
    "Squats train the quadriceps,glutes,and core stability",
    "To gain high muscle,you need enough protien,calorie surplus,and progressive overload"

]
def encode_texts(texts):
    inputs=embed_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs=embed_model(**inputs)

    token_embeddings=outputs.last_hidden_state
    attention_mask=inputs["attention_mask"]

    mask=attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    masked_embeddings=token_embeddings*mask

    sum_embeddings=torch.sum(masked_embeddings,dim=1)
    sum_mask=torch.clamp(mask.sum(dim=1),min=1e-9)

    sentence_embeddings=sum_embeddings/sum_mask
    sentence_embeddings=F.normalize(sentence_embeddings,p=2,dim=1)

    return sentence_embeddings
doc_embeddings=encode_texts(documents)
def retrieve_doc(query,documents,doc_embeddings,top_k=2):
    query_embeddings=encode_texts([query])
    scores=torch.matmul(query_embeddings,doc_embeddings.T)
    topk_values,topk_indices=torch.topk(scores,k=top_k,dim=1)
    retrieved_doc=[]
    for i in range(top_k):
        idx=topk_indices[0][i].item()
        score=topk_values[0][i].item()
        retrieved_doc.append({
            "doc_index":idx,
            "document":documents[idx],
            "score":score
        })
    return retrieved_doc
def build_baseline_prompt(query,retrieved_doc):
    context="\n".join(
        [f"-{item['document']}" for item in retrieved_doc]
    )
    prompt=(
        "Answer the question using the context\n\n"
        f"Context:\n{context}\n\n"
        f"Qusetion:{query}\n"
        "Answer:"
    )
    return prompt
def build_optimized_prompt(query,retrieved_doc):
    context="\n".join(
        [f"{i+1}.{item['document']}" for i,item in enumerate(retrieved_doc)]
    )
    prompt=(
        "You are a helpful AI assistant.\n"
        "Answer the qusetion using only provided context.\n"
        "Use the most relevant context sentence.\n"
        "Keep imformation exercise names,food terms,and nutrition terms in the answer.\n"
        "Do not add unrelated details\n"
        f"Context:{context}\n\n"
        f"Question:{query}\n"
        "Answer:"
    )
    return prompt
def generate_answer(prompt):
    inputs=gen_tokenizer(prompt,truncation=True,return_tensors="pt")
    with torch.no_grad():
        outputs_ids=gen_model.generate(
            **inputs,
            max_new_tokens=64
        )
    answer=gen_tokenizer.decode(outputs_ids[0],skip_special_tokens=True)
    return answer
def evaluate_answer(answer,expected_keywards):
    answer_lower=answer.lower()
    matched=0
    for keyward in expected_keywards:
        if keyward.lower() in answer_lower:
            matched += 1
    keyward_score=matched/len(expected_keywards)
    print("answer_lower",answer_lower)
    print("expected_keyward",expected_keywards)
    return keyward_score,matched
def compare_prompts(query,documents,doc_embeddings,expected_keywards,top_k=2):
    retrieved_doc=retrieve_doc(query,documents,doc_embeddings,top_k=top_k)
    
    baseline_prompt=build_baseline_prompt(query,retrieved_doc)
    baseline_answer=generate_answer(baseline_prompt)

    optimized_prompt=build_optimized_prompt(query,retrieved_doc)
    optimized_answer=generate_answer(optimized_prompt)

    baseline_score,baseline_matched=evaluate_answer(baseline_answer,expected_keywards)
    optimized_score,optimized_matched=evaluate_answer(optimized_answer,expected_keywards)

    return{
        "query":query,
        "retrieved_docs":retrieved_doc,
        "baseline_prompt":baseline_prompt,
        "baseline_answer":baseline_answer,
        "optimized_prompt":optimized_prompt,
        "optimized_answer":optimized_answer,
        "baseline_score":baseline_score,
        "baseline_matched":baseline_matched,
        "optimized_score":optimized_score,
        "optimized_matched":optimized_matched
    }
test_cases=[
    {
        "query":"What should I eat for dinner during fat loss?",
        "expected_keywards":["high protein","low fat","moderate carbs"]
    },
    {
        "query":"What exercises should I do for chest day",
        "expected_keywards":["bench press","incline dumbbell press"]
    },
    {
        "query":"How do I eat during muscle gain",
        "expected_keywards":["protein","calorie surplus"]
    }
]
total_baseline_score=0
total_optimized_score=0
for i,case in enumerate(test_cases):
    result=compare_prompts(case["query"],documents,doc_embeddings,case["expected_keywards"],top_k=2)
    total_baseline_score += result["baseline_score"]
    total_optimized_score += result["optimized_score"]

    print(f"\n===== Case {i+1} =====")
    print("Query:", result["query"])

    print("\nRetrieved docs:")
    for rank,item in enumerate(result["retrieved_docs"]):
        print(
            f"Rank {rank+1}:doc_index={item['doc_index']},"
            f"score={round(item['score'],4)}"
        )
        print(item["document"])
    print("\nBaseline answer:")
    print(result["baseline_answer"])
    print(
        "Baseline matched:",
        f"{result['baseline_matched']}/{len(case['expected_keywards'])}"
    )
    print("Baseline score:",round(result["baseline_score"],4))
    print("\nOptimized answer:")
    print(result["optimized_answer"])
    print(
        "Optimized matched:",
        f"{result['optimized_matched']}/{len(case['expected_keywards'])}"
    )
    print("Optimized score:",round(result["optimized_score"],4))
avg_baseline_score=total_baseline_score/len(test_cases)
avg_optimized_score=total_optimized_score/len(test_cases)
print("\n===== Final Prompt Comparison Summary =====")
print("Average Baseline Score:",round(avg_baseline_score,4))
print("Average optimized Score:",round(avg_optimized_score,4))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6549.90it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 7722.22it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


answer_lower high protien,low fat,and moderate carbs
expected_keyward ['high protein', 'low fat', 'moderate carbs']
answer_lower high protien,low fat,and moderate carbs
expected_keyward ['high protein', 'low fat', 'moderate carbs']

===== Case 1 =====
Query: What should I eat for dinner during fat loss?

Retrieved docs:
Rank 1:doc_index=0,score=0.8199
For fat loss dinner,choose high protien,low fat,and moderate carbs
Rank 2:doc_index=3,score=0.2736
To gain high muscle,you need enough protien,calorie surplus,and progressive overload

Baseline answer:
high protien,low fat,and moderate carbs
Baseline matched: 2/3
Baseline score: 0.6667

Optimized answer:
high protien,low fat,and moderate carbs
Optimized matched: 2/3
Optimized score: 0.6667
answer_lower good chest exercises include bench press,incline dumbbell press,and cable fly
expected_keyward ['bench press', 'incline dumbbell press']
answer_lower bench press,incline dumbbell press,and cable fly
expected_keyward ['bench press', 'incline

In [5]:
"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
"google/flan-t5-small"
"For fat loss dinner,choose high protien,low fat,and moderate carbs"
"Good chest exercises include bench press,incline dumbbell press,and cable fly"
"Squats train the quadriceps,glutes,and core stability"
"To gain high muscle,you need enough protien,calorie surplus,and progressive overload"

"Answer the question using the context\n\n"
f"Context:\n{context}\n\n"
f"Qusetion:{query}\n"
"Answer:"

"You are a helpful AI assistant.\n"
"Answer the qusetion using only provided context.\n"
"If the context contains the answer,give a short and clear reply.\n"
"Focus on the most relevant information only.\n"
"Do not add unrelated details\n"
"Do not repeat the full context\n"
f"Context:{context}\n\n"
f"Question:{query}\n"
"Answer in one short sentence:"

test_cases=[
    {
        "query":"What should I eat for dinner during fat loss?",
        "expected_keywards":["high protein","low fat","moderate carbs"]
    },
    {
        "query":"What exercises should I do for chest day",
        "expected_keywards":["bench press","incline dumbbell press"]
    },
    {
        "query":"How do I eat during muscle gain",
        "expected_keywards":["protein","calorie surplus"]
    }
]

NameError: name 'context' is not defined